# Run the full pipeline (raw → identity figures)

Idempotent orchestrator: runs each missing step
(cells → REDSEA → RESTORE → RESTORE efficacy (mxnorm) → lineage → QuPath export → identity figures) and
prints a status table. Skips steps whose outputs already exist unless `force=True`. This mirrors the
individual step notebooks 01–06 (+ `03b` for the RESTORE efficacy diagnostic).

In [ ]:
# Parameters for this step (self-contained -- no config.ini). Edit the paths for your machine.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # make `phenocycler` importable from notebooks/
from phenocycler import PipelineConfig

REPO = pathlib.Path.cwd().resolve().parents[1]        # Islet-Explorer-Senior (parent of the submodule; data/ lives here)
# Run on a subset of donors (None = every donor under data/cells/donor_id=*).
DONORS = None            # e.g. ["6374", "6380"] to iterate on a few


cfg = PipelineConfig(
    # --- Paths (edit for your machine) ---
    data_dir       = REPO / "data",
    images_dir     = pathlib.Path("/home/smith6jt/IO60panc2nd/Images"),
    cells_csv      = pathlib.Path("/home/smith6jt/IO60panc2nd/Cellmeasurements.csv"),
    donor_metadata = pathlib.Path("/home/smith6jt/IO60panc2nd/donor_metadata_panc.xlsx"),
    # --- REDSEA (step 2) ---
    redsea_downsample=1.0, redsea_edge_radius=0, redsea_comp_mode=0, redsea_alpha=1.0, redsea_gap_bridge=1,
    # --- RESTORE (step 3) ---
    restore_model="SSC", restore_subsample=15000, restore_robust=True, restore_robust_factor=3.0,
    restore_min_cell_area=5.0, restore_seed=0, restore_idx_floor_q=0.5,
    # --- RESTORE efficacy diagnostic (step 3b) ---
    restore_diag_subsample=20000, restore_diag_seed=0,
    # --- compute ---
    n_jobs=8, duckdb_threads=8, use_gpu=False,
)
print("data_dir     :", cfg.data_dir)
print("n_jobs       :", cfg.n_jobs, "| duckdb_threads:", cfg.duckdb_threads, "| use_gpu:", cfg.use_gpu)
donors = DONORS or cfg.discover_donors()
print(f"donors: {len(donors)} " + ("(subset)" if DONORS else "(all)") + f" -> {donors[:6]}" + (" ..." if len(donors) > 6 else ""))


In [ ]:
from phenocycler.pipeline import run_pipeline, print_status
print_status(cfg)   # what already exists

In [ ]:
# NOTE: the orchestrator runs ALL donors; use the per-step notebooks 02/03/04/06 to run on a DONORS subset.
# Run everything missing. Use only=[...] to run a subset, force=True to redo.
run_pipeline(cfg, force=False)